# Differential Privacy: Formal Privacy Guarantees

Differential privacy provides a mathematical definition of privacy and a framework for building mechanisms that satisfy it. Unlike ad-hoc anonymization, DP gives quantifiable guarantees that hold even against adversaries with arbitrary auxiliary information.

## The DP Definition

A randomized mechanism M is (ε, δ)-differentially private if for all datasets D and D' differing in one record, and all outputs S:

```
Pr[M(D) ∈ S] ≤ exp(ε) * Pr[M(D') ∈ S] + δ
```

**ε (epsilon)**: privacy budget. Lower = stronger privacy. ε < 1 is strong; ε = 10 is weak; ε = ln(2) ≈ 0.69 is a common target.

**δ (delta)**: failure probability. Set to 1/n² where n is dataset size. Represents the probability that the guarantee fails entirely.

**Intuition**: any one person's data cannot change the output distribution much. An adversary who sees the output learns almost as much as they would have without that person's data.

In [ ]:
import math, random

def laplace_mechanism(true_value: float, sensitivity: float, epsilon: float) -> float:
    scale = sensitivity / epsilon
    rng = random.Random()
    u = rng.uniform(-0.5, 0.5)
    return true_value - scale * (1 if u >= 0 else -1) * math.log(1 - 2*abs(u))

def gaussian_mechanism(true_value: float, sensitivity: float, epsilon: float, delta: float = 1e-5) -> float:
    sigma = math.sqrt(2 * math.log(1.25 / delta)) * sensitivity / epsilon
    return true_value + random.gauss(0, sigma)

def clamp(value, lo, hi):
    return max(lo, min(hi, value))

def dp_mean(data: list, epsilon: float, delta: float, lower: float, upper: float) -> float:
    n = len(data)
    clamped = [clamp(x, lower, upper) for x in data]
    true_mean = sum(clamped) / n
    # Sensitivity of mean with bounded inputs: (upper-lower)/n
    sensitivity = (upper - lower) / n
    return gaussian_mechanism(true_mean, sensitivity, epsilon, delta)

# Demonstrate privacy-utility tradeoff
rng = random.Random(42)
true_data = [rng.gauss(50, 10) for _ in range(1000)]
true_mean = sum(true_data) / len(true_data)

print('Privacy-utility tradeoff for mean estimation:')
print(f'True mean: {true_mean:.4f}')
print(f'{'Epsilon':>10} {'Delta':>10} {'DP Mean':>10} {'Error':>8}')
print('-' * 45)
for eps in [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]:
    rng2 = random.Random(42)
    random.seed(42)
    dp_est = dp_mean(true_data, eps, 1e-5, lower=0, upper=100)
    error = abs(dp_est - true_mean)
    print(f'{eps:>10.1f} {1e-5:>10.1e} {dp_est:>10.4f} {error:>8.4f}')

## Privacy Composition

DP mechanisms compose: if you run k algorithms each with privacy budget ε, the total budget is at most k*ε (basic composition). This matters for training: each gradient step consumes a bit of privacy budget.

**Rényi Differential Privacy (RDP)** provides tighter composition bounds than basic DP. It's defined in terms of Rényi divergence and converts to (ε, δ)-DP at the end of training.

**Privacy Loss Random Variable (PRV)**: exact composition via numerical convolution of privacy loss distributions. Used in modern DP-SGD implementations (PRV accountant in Opacus).

The key insight: with tight accounting, you can train longer (more gradient steps) for the same privacy budget than naive composition would suggest.

In [ ]:
def basic_composition(epsilon_i: float, k: int) -> float:
    return k * epsilon_i

def advanced_composition(epsilon_i: float, k: int, delta: float = 1e-5) -> float:
    # Advanced composition theorem
    return math.sqrt(2 * k * math.log(1/delta)) * epsilon_i + k * epsilon_i * (math.exp(epsilon_i) - 1)

def rdp_to_dp(alpha: float, rdp_epsilon: float, delta: float = 1e-5) -> float:
    return rdp_epsilon + math.log((alpha - 1) / alpha) - (math.log(delta) + math.log(alpha)) / (alpha - 1)

print('Composition: privacy budget after k gradient steps (per-step ε=0.1):')
print(f'{'k steps':>10} {'Basic':>12} {'Advanced':>12}')
print('-' * 38)
for k in [10, 100, 1000, 10000]:
    basic = basic_composition(0.1, k)
    adv = advanced_composition(0.1, k)
    print(f'{k:>10} {basic:>12.2f} {adv:>12.4f}')